In [ ]:
import pandas as pd
from pathlib import Path
import os
import openpyxl
import re
import json
import pandas as pd


In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SAMPLES =  PROJECT_ROOT / "data" / "samples"
BENCHMARKS_FINAL = PROJECT_ROOT / "benchmarks" / "final"

In [ ]:
cols = ["transcript_id", "question_order", "output"]

llama_8b_0temp = pd.read_csv(
    BENCHMARKS_FINAL / "llama_8b_0.csv",
    usecols=cols,
    engine="pyarrow"
)

llama_8b_1temp = pd.read_csv(
    BENCHMARKS_FINAL / 'llama_8b_dot1.csv',
    usecols=cols,
    engine="pyarrow"
)

llama_8b_2temp = pd.read_csv(
    BENCHMARKS_FINAL / 'llama_8b_dot2.csv',
    usecols=cols,
    engine="pyarrow"
)

llama_8b_3temp = pd.read_csv(
    BENCHMARKS_FINAL / 'llama_8b_dot3.csv',
    usecols=cols,
    engine="pyarrow"
)

llama_8b_5temp = pd.read_csv(
    BENCHMARKS_FINAL / 'llama_8b_dot5.csv',
    usecols=cols,
    engine="pyarrow"
)


In [ ]:
for i in [llama_8b_0temp, llama_8b_1temp, llama_8b_2temp, llama_8b_3temp, llama_8b_5temp]:
    i["question_order"] = i["question_order"].astype(int)
    print(i.columns)
    print(len(i))   

In [ ]:
llama_8b_0temp = llama_8b_0temp.rename(columns={"output": "outputllama-3.1-8b_0temp"})
llama_8b_1temp = llama_8b_1temp.rename(columns={"output": "outputllama-3.1-8b_1temp"})
llama_8b_2temp = llama_8b_2temp.rename(columns={"output": "outputllama-3.1-8b_2temp"})
llama_8b_3temp = llama_8b_3temp.rename(columns={"output": "outputllama-3.1-8b_3temp"})
llama_8b_5temp = llama_8b_5temp.rename(columns={"output": "outputllama-3.1-8b_5temp"})

In [ ]:
for name, df in [
    ("llama_8b_0temp", llama_8b_0temp),
    ("llama_8b_1temp", llama_8b_1temp),
    ("llama_8b_2temp", llama_8b_2temp),
    ("llama_8b_3temp", llama_8b_3temp),
    ("llama_8b_5temp", llama_8b_5temp),
]:
    dupes = df.duplicated(subset=["transcript_id", "question_order"]).sum()
    print(f"{name}: {dupes} duplicates")

In [ ]:
merge = (
    llama_8b_0temp
    .merge(llama_8b_1temp, on=["transcript_id", "question_order"])
    .merge(llama_8b_2temp, on=["transcript_id", "question_order"])
    .merge(llama_8b_3temp, on=["transcript_id", "question_order"])
    .merge(llama_8b_5temp, on=["transcript_id", "question_order"]))

del  llama_8b_0temp, llama_8b_1temp, llama_8b_2temp, llama_8b_3temp, llama_8b_5temp

In [ ]:
models = [
    ("outputllama-3.1-8b_0temp", "llama_8b_0temp"),
    ("outputllama-3.1-8b_1temp", "llama_8b_1temp"),
    ("outputllama-3.1-8b_2temp", "llama_8b_2temp"),
    ("outputllama-3.1-8b_3temp", "llama_8b_3temp"),
    ("outputllama-3.1-8b_5temp", "llama_8b_5temp"),
]

In [ ]:
def extract_fields(row, col, source):
    text = row[col]

    if pd.isna(text):
        return None

    try:
        x = json.loads(text)

        return {
            "transcript_id": row["transcript_id"],
            "source": source,
            "fli": x.get("forward_looking_intensity"),
            "spec": x.get("specificity"),
            "sub": x.get("economic_substance"),
            "tone": x.get("tone"),
            "cert": x.get("certainty"),
            "main_focus": x.get("context_summary", {}).get("main_focus"),
            "secondary_focus": x.get("context_summary", {}).get("secondary_focus"),
            "managerial_horizon": x.get("context_summary", {}).get("managerial_horizon"),
            "overall_outlook": x.get("context_summary", {}).get("overall_outlook"),
            "question_order": row["question_order"] ,
        }

    except:
        return None

In [ ]:
merge.columns

In [ ]:
import time

all_results = []

for col, name in models:
    start = time.time()

    temp = merge.apply(lambda row: extract_fields(row, col, name), axis=1)
    temp = temp.dropna().tolist()

    all_results.extend(temp)

    end = time.time()

    print(f"{name} | rows: {len(temp)} | time: {end - start:.2f}s")

    del temp

In [ ]:
df_final = pd.DataFrame(all_results)

In [ ]:
df_final.columns

In [ ]:
llm_long = df_final.melt(
    id_vars=["transcript_id", "question_order", "source"],
    value_vars=["fli", "spec", "sub", "tone", "cert", "main_focus", "secondary_focus", "managerial_horizon", "overall_outlook"],
    var_name="Measure",
    value_name="Score"
)

In [ ]:
own = pd.read_excel(BENCHMARKS_FINAL / "output_own.xlsx")



In [ ]:
own.columns

In [ ]:
rename_map = {
    "fli": "FLI",
    "spec": "Specificity",
    "sub": "EC Sub",
    "tone": "Tone",
    "cert": "Certainty",
    "main_focus": "Main Foc",
    "secondary_focus": "Second Foc",
    "managerial_horizon": "Horizon",
    "overall_outlook": "Outlook"
}
# Apply ONLY to raw names, keep existing correct ones
llm_long["Measure"] = llm_long["Measure"].replace(rename_map)

In [ ]:
numeric_measures = ["FLI", "Specificity", "EC Sub", "Tone", "Certainty"]

df_num = llm_long[llm_long["Measure"].isin(numeric_measures)].copy()

df_num["Score"] = pd.to_numeric(df_num["Score"], errors="coerce")

pivot = df_num.pivot_table(
    index=["transcript_id", "Measure"],
    columns="source",
    values="Score"
)

pivot = pivot.dropna()

pivot.corr()

In [ ]:
print(llm_long["Measure"].value_counts())

In [ ]:
llm_num = llm_long[llm_long["Measure"].isin(numeric_measures)].copy()

llm_num["Score"] = pd.to_numeric(llm_num["Score"], errors="coerce")

llm_num = llm_num.rename(columns={"Score": "Score_llm"})

In [ ]:
own_num = own[own["Measure"].isin(numeric_measures)].copy()

own_num["Score"] = pd.to_numeric(own_num["Score"], errors="coerce")

own_num = own_num.rename(columns={
    "Score": "Score_manual",
    "Source": "source_manual"
})



In [ ]:
merged = llm_num.merge(
    own_num,
    on=["transcript_id", "question_order", "Measure"]
)

In [ ]:
merged.columns

In [ ]:
merged[["Score_llm", "Score_manual"]].corr()

In [ ]:
corrs_model = (
    merged
    .groupby(["source", "Measure"], group_keys=False)
    .apply(lambda x: x["Score_llm"].corr(x["Score_manual"]))
    .unstack()
)
print(corrs_model)

In [ ]:
desc = (
    merged.groupby(["source", "Measure"])["Score_llm"]
      .describe()
      .reset_index()
)

print(desc)


desc.to_csv(DATA_PROCESSED/"llm_score_descriptions_temperatures.csv", index=False)